# Advanced Alzheimer (AD) diversity analysis

**Author: Stephany Z. Flores-Ramos (sflores@health.ucsd.edu)**

In [1]:
import qiime2
import biom
import pandas as pd
import numpy as np
import time
import seaborn as sns
import matplotlib.pyplot as plt
import os

%matplotlib inline
os.chdir('/mnt/zarrinpar/scratch/sfloresr/AD_EM/')

**first we need to clean the metadata file to run through GNPS to get FBMN**

In [9]:
mtb_md=pd.read_csv('./data_files/LCMS/metadata/AD_TRF_ReDu_MD.txt',sep='\t',index_col=0).reset_index()
mouse_md=pd.read_csv('./data_files/LCMS/metadata/AlzheimersTRF_fecesandileum_metabolomics_metadata_10.25.22.txt',sep='\t',index_col=0).reset_index().rename(columns={"sample_name":"SubjectIdentifierAsRecorded"})
md=pd.merge(mtb_md,mouse_md, on='SubjectIdentifierAsRecorded', how='left').set_index('filename')
md

,SampleType,SampleTypeSub1,NCBITaxonomy,YearOfAnalysis,SampleCollectionMethod,SampleExtractionMethod,InternalStandardsUsed,MassSpectrometer,IonizationSourceAndPolarity,ChromatographyAndPhase,...,env_material,env_package,geo_loc_name,host_common_name,host_scientific_name,host_taxid,iacuc_institute,iacuc_protocol_id,life_stage,title
filename,,,,,,,,,,,,,,,,,,,,,
P1_A1_315_4.mzML,animal,biofluid,10090|Mus musculus,2022,feces,methanol-water (1:1),sulfamethazine,Q Exactive|MS:1001911,electrospray ionization (positive),reverse phase (C18),...,feces,host-associated,USA:CA: San Diego,Mouse,Mus musculus,10090.0,UCSD,S16285,adult,ALZ/TRF
P1_A2_331_4.mzML,animal,biofluid,10090|Mus musculus,2022,feces,methanol-water (1:1),sulfamethazine,Q Exactive|MS:1001911,electrospray ionization (positive),reverse phase (C18),...,feces,host-associated,USA:CA: San Diego,Mouse,Mus musculus,10090.0,UCSD,S16285,adult,ALZ/TRF
P1_A3_310_4.mzML,animal,biofluid,10090|Mus musculus,2022,feces,methanol-water (1:1),sulfamethazine,Q Exactive|MS:1001911,electrospray ionization (positive),reverse phase (C18),...,feces,host-associated,USA:CA: San Diego,Mouse,Mus musculus,10090.0,UCSD,S16285,adult,ALZ/TRF
P1_A4_302_8.mzML,animal,biofluid,10090|Mus musculus,2022,feces,methanol-water (1:1),sulfamethazine,Q Exactive|MS:1001911,electrospray ionization (positive),reverse phase (C18),...,feces,host-associated,USA:CA: San Diego,Mouse,Mus musculus,10090.0,UCSD,S16285,adult,ALZ/TRF
P1_A5_327_8.mzML,animal,biofluid,10090|Mus musculus,2022,feces,methanol-water (1:1),sulfamethazine,Q Exactive|MS:1001911,electrospray ionization (positive),reverse phase (C18),...,feces,host-associated,USA:CA: San Diego,Mouse,Mus musculus,10090.0,UCSD,S16285,adult,ALZ/TRF
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
P2_H8_127_T1.mzML,animal,tissue,10090|Mus musculus,2022,"solid material, frozen",methanol-water (1:1),sulfamethazine,Q Exactive|MS:1001911,electrospray ionization (positive),reverse phase (C18),...,ileum,host-associated,USA:CA: San Diego,Mouse,Mus musculus,10090.0,UCSD,S16285,adult,ALZ/TRF
P2_H9_206_T1.mzML,animal,tissue,10090|Mus musculus,2022,"solid material, frozen",methanol-water (1:1),sulfamethazine,Q Exactive|MS:1001911,electrospray ionization (positive),reverse phase (C18),...,ileum,host-associated,USA:CA: San Diego,Mouse,Mus musculus,10090.0,UCSD,S16285,adult,ALZ/TRF
P2_H10_220_T1.mzML,animal,tissue,10090|Mus musculus,2022,"solid material, frozen",methanol-water (1:1),sulfamethazine,Q Exactive|MS:1001911,electrospray ionization (positive),reverse phase (C18),...,ileum,host-associated,USA:CA: San Diego,Mouse,Mus musculus,10090.0,UCSD,S16285,adult,ALZ/TRF


In [10]:
#save the new metadata file
md.to_csv('./data_files/LCMS/metadata/AD_TRF_ReDu_MD_ext.txt',sep='\t')

we will use this metadata file when running the files through GNPS

__________

## **Cleaning the FBMN outputs to remove all the blanks**

Using R, I removed all blanks from the feature table and metadata from the FBMN output

In [ ]:
biom convert \
  -i /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-noblanks.txt \
  -o /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-noblanks.biom \
  --table-type="OTU table" \
  --to-hdf5

! qiime tools import \
  --input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-noblanks.biom \
  --output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-noblanks.qza \
  --type FeatureTable[Frequency]

## **Calculating the alpha diversity**

To do this we use the qiime plugin to calculate shannon diversity, observed OTUs, and Faith's PD. The latter we generated using the phylogenetic tree 'qemistree_serum.qza' generated through qemistree.

In [ ]:
#The alpha diveristy using 'shannon' 
! qiime diversity alpha \
--i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-noblanks.qza \
--p-metric shannon \
--o-alpha-diversity /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/alpha_shannon_lcms.qza

In [ ]:
! qiime tools export \
--input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/alpha_shannon_lcms.qza \
--output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/alpha_shannon_lcms

In [ ]:
#The alpha diveristy using 'observed_feature'  
! qiime diversity alpha \
--i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-noblanks.qza \
--p-metric observed_features \
--o-alpha-diversity /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/alpha_obsOTU_lcms.qza

In [ ]:
! qiime tools export \
--input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/alpha_obsOTU_lcms.qza \
--output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/alpha_obsOTU_lcms

## **Calculating the beta diversity**
To do this we used the qiime plugin to calculate bray curtis, aitchison, weighted and unweighted unifrac. The latter two were generated using the phylogenetic tree qemistree_serum.qza generated through qemistree.

In [ ]:
#The beta diversity using 'bray curtis'  
! qiime diversity beta \
--i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-noblanks.qza \
--p-metric braycurtis \
--o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_braycurtis_lcms.qza

In [ ]:
! qiime tools export \
--input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_braycurtis_lcms.qza \
--output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_braycurtis_lcms

In [ ]:
#The beta diversity using 'aitchison'  
! qiime diversity beta \
--i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-noblanks.qza \
--p-metric aitchison \
--o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_aitchison_lcms.qza

In [ ]:
! qiime tools export \
--input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_aitchison_lcms.qza \
--output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_aitchison_lcms

### Deicode (to calculate aitchison)

In additon to calculating these metric through the default plugin methods, I also wanted to use this new one that uses log abundances to account for sparse mtbs as the previous methods do not correct for this (very possible) bias.

In [ ]:
! qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-noblanks.qza \
    --p-min-feature-count 10 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_deicode_ordination.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_deicode_lcms.qza

In [ ]:
! qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_deicode_ordination.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-noblanks.txt \
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_deicode_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_deicode_lcms.qza  \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-noblanks.txt \
    --m-metadata-column ATTRIBUTE_BodyPartName \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_deicode_bodypart-significance.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_deicode_lcms.qza  \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-noblanks.txt \
    --m-metadata-column ATTRIBUTE_ZT_collection_time \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_deicode_ZTtime-significance.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_deicode_lcms.qza  \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-noblanks.txt \
    --m-metadata-column ATTRIBUTE_geno_n_feedingcond \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/combined/beta_deicode_genofeeding-significance.qzv

### Deicode (to calculate aitchison) Fecal Only

In [ ]:
! biom convert \
  -i /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal.txt \
  -o /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal.biom \
  --table-type="OTU table" \
  --to-hdf5

! qiime tools import \
  --input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal.biom \
  --output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal.qza \
  --type FeatureTable[Frequency]

In [ ]:
! qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal.qza \
    --p-min-feature-count 10 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_ordination_fecal.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal_lcms.qza

In [ ]:
! qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_ordination_fecal.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal.txt \
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal.txt \
    --m-metadata-column ATTRIBUTE_geno_n_feedingcond \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal_feedingcond-significance.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal.txt \
    --m-metadata-column ATTRIBUTE_ZT_collection_time \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal_ZT-significance.qzv

run CTF for the time series data

In [ ]:
qiime gemelli ctf\
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal.qza  \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal.txt \
    --m-feature-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/DB_analogresult/feature_annotations_clean.txt \
    --p-state-column ZT_time\
    --p-individual-id-column ATTRIBUTE_host_subject_id\
    --output-dir /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults

qiime longitudinal volatility \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/state_subject_ordination.qza \
    --p-state-column ZT_time \
    --p-individual-id-column subject_id \
    --p-default-group-column ATTRIBUTE_geno_n_feedingcond \
    --p-default-metric PC1 \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/state_subject_ordination.qzv

In [24]:
import pandas as pd
from qiime2 import Metadata

# first we import the metdata into pandas
mf = pd.read_csv('./data_files/FBMN/metadata_table/metadata_table-fecal.txt', sep='\t',index_col=0)
# next we aggregate by subjects (i.e. 'host_subject_id') 
# and keep the first instance of 'diagnosis_full' by subject.
mf = mf.groupby('ATTRIBUTE_host_subject_id').agg({'ATTRIBUTE_geno_n_feedingcond':'first','ATTRIBUTE_sex':'first'})
# now we save the metadata in QIIME2 format.
mf.index.name = '#SampleID'
mf.to_csv('./data_files/FBMN/metadata_table/subject-metadata_table-fecal.tsv', sep='\t')


In [ ]:
qiime emperor biplot\
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/subject_biplot.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/subject-metadata_table-fecal.tsv \
    --m-feature-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/DB_analogresult/feature_annotations_clean.txt \
    --p-number-of-features 100\
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/subject_biplot.qzv

qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/distance_matrix.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal.txt \
    --m-metadata-column ATTRIBUTE_geno_n_feedingcond \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/CTFsubject_fecal_genofeed-significance.qzv

qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/distance_matrix.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal.txt \
    --m-metadata-column ATTRIBUTE_sex \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/CTFsubject_fecal_sex-significance.qzv

In [ ]:
qiime qurro loading-plot\
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal.qza\
    --i-ranks /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/subject_biplot.qza\
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal.txt\
    --m-feature-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/DB_analogresult/feature_annotations_clean.txt\
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/qurro.qzv

In [ ]:
qiime longitudinal volatility \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/metadata_table-fecal-PC1-NatLog.txt\
    --p-state-column ZT_time \
    --p-individual-id-column ATTRIBUTE_host_subject_id \
    --p-default-group-column ATTRIBUTE_geno_n_feedingcond \
    --p-default-metric Current_Natural_Log_Ratio \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/PC1_log_ratio_plot.qzv

qiime longitudinal volatility \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/metadata_table-fecal-PC2-NatLog.txt\
    --p-state-column ZT_time \
    --p-individual-id-column ATTRIBUTE_host_subject_id \
    --p-default-group-column ATTRIBUTE_geno_n_feedingcond \
    --p-default-metric Current_Natural_Log_Ratio \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/PC2_log_ratio_plot.qzv

qiime longitudinal volatility \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/metadata_table-fecal-PC3-NatLog.txt\
    --p-state-column ZT_time \
    --p-individual-id-column ATTRIBUTE_host_subject_id \
    --p-default-group-column ATTRIBUTE_geno_n_feedingcond \
    --p-default-metric Current_Natural_Log_Ratio \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/PC3_log_ratio_plot.qzv

In [ ]:
qiime longitudinal linear-mixed-effects\
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/metadata_table-fecal-PC1-NatLog.txt\
    --p-state-column ZT_time \
    --p-individual-id-column ATTRIBUTE_host_subject_id \
    --p-group-columns ATTRIBUTE_geno_n_feedingcond \
    --p-metric Current_Natural_Log_Ratio \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/PC1_lme_log_ratio.qzv

qiime longitudinal linear-mixed-effects\
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/metadata_table-fecal-PC2-NatLog.txt\
    --p-state-column ZT_time \
    --p-individual-id-column ATTRIBUTE_host_subject_id \
    --p-group-columns ATTRIBUTE_geno_n_feedingcond \
    --p-metric Current_Natural_Log_Ratio \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/PC2_lme_log_ratio.qzv

qiime longitudinal linear-mixed-effects\
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/metadata_table-fecal-PC3-NatLog.txt\
    --p-state-column ZT_time \
    --p-individual-id-column ATTRIBUTE_host_subject_id \
    --p-group-columns ATTRIBUTE_geno_n_feedingcond \
    --p-metric Current_Natural_Log_Ratio \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/fecal_CTFresults/PC3_lme_log_ratio.qzv

### Deicode (to calculate aitchison) Fecal Only ZT0

In [ ]:
! biom convert \
  -i /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal_ZT0.txt \
  -o /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal_ZT0.biom \
  --table-type="OTU table" \
  --to-hdf5

! qiime tools import \
  --input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal_ZT0.biom \
  --output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal_ZT0.qza \
  --type FeatureTable[Frequency]

In [ ]:
! qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal_ZT0.qza \
    --p-min-feature-count 10 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_ordination_fecal-ZT0.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-ZT0.qza

In [ ]:
! qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_ordination_fecal-ZT0.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal-ZT0.txt\
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-ZT0_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-ZT0.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal-ZT0.txt \
    --m-metadata-column ATTRIBUTE_geno_n_feedingcond \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-ZT0-genofeed_significance.qzv

### Deicode (to calculate aitchison) Fecal Only Tg-TRF

In [ ]:
! biom convert \
  -i /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-tgTRF.txt \
  -o /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-tgTRF.biom \
  --table-type="OTU table" \
  --to-hdf5

! qiime tools import \
  --input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-tgTRF.biom \
  --output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-tgTRF.qza \
  --type FeatureTable[Frequency]

In [ ]:
! qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-tgTRF.qza \
    --p-min-feature-count 10 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_ordination_fecal-tgTRF.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgTRF_lcms.qza

In [ ]:
! qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_ordination_fecal-tgTRF.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal-tgTRF.txt\
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgTRF_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgTRF_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal.txt \
    --m-metadata-column ATTRIBUTE_ZT_collection_time \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgTRF-ZT-significance.qzv

qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgTRF_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal.txt \
    --m-metadata-column BiologicalSex \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgTRF-sex-significance.qzv

### Deicode (to calculate aitchison) Fecal Only Tg-ALF

In [ ]:
! biom convert \
  -i /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-tgALF.txt \
  -o /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-tgALF.biom \
  --table-type="OTU table" \
  --to-hdf5

! qiime tools import \
  --input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-tgALF.biom \
  --output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-tgALF.qza \
  --type FeatureTable[Frequency]

In [ ]:
! qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-tgALF.qza \
    --p-min-feature-count 10 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_ordination_fecal-tgALF.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgALF_lcms.qza

In [ ]:
! qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_ordination_fecal-tgALF.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal-tgALF.txt\
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgALF_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgALF_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal-tgALF.txt \
    --m-metadata-column ATTRIBUTE_ZT_collection_time \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgALF-ZT-significance.qzv

qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgALF_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal-tgALF.txt \
    --m-metadata-column BiologicalSex \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-tgALF-sex-significance.qzv

### Deicode (to calculate aitchison) Fecal Only NTg-ALF

In [ ]:
! biom convert \
  -i /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-ntgALF.txt \
  -o /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-ntgALF.biom \
  --table-type="OTU table" \
  --to-hdf5

! qiime tools import \
  --input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-ntgALF.biom \
  --output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-ntgALF.qza \
  --type FeatureTable[Frequency]

In [ ]:
! qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-fecal-ntgALF.qza \
    --p-min-feature-count 10 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_ordination_fecal-ntgALF.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-ntgALF_lcms.qza

In [ ]:
! qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_ordination_fecal-ntgALF.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal-ntgALF.txt\
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-ntgALF_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-ntgALF_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal-ntgALF.txt \
    --m-metadata-column ATTRIBUTE_ZT_collection_time \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-ntgALF-ZT-significance.qzv

qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-ntgALF_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-fecal-ntgALF.txt \
    --m-metadata-column BiologicalSex \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/fecal/beta_deicode_fecal-ntgALF-sex-significance.qzv

### Deicode (to calculate aitchison) Ileum Only

In [ ]:
! biom convert \
  -i /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum.txt \
  -o /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum.biom \
  --table-type="OTU table" \
  --to-hdf5

! qiime tools import \
  --input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum.biom \
  --output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum.qza \
  --type FeatureTable[Frequency]

In [ ]:
! qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum.qza \
    --p-min-feature-count 10 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ordination_ileum.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum_lcms.qza

In [ ]:
! qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ordination_ileum.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-ileum.txt \
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-ileum.txt \
    --m-metadata-column ATTRIBUTE_geno_n_feedingcond \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum_feedingcond-significance.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-ileum.txt \
    --m-metadata-column ATTRIBUTE_ZT_collection_time \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum_ZT-significance.qzv

### Deicode (to calculate aitchison) Ileum Only Tg-TRF

In [ ]:
! biom convert \
  -i /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-tgTRF.txt \
  -o /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-tgTRF.biom \
  --table-type="OTU table" \
  --to-hdf5

! qiime tools import \
  --input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-tgTRF.biom \
  --output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-tgTRF.qza \
  --type FeatureTable[Frequency]

In [ ]:
! qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-tgTRF.qza \
    --p-min-feature-count 10 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ordination_ileum-tgTRF.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-tgTRF_lcms.qza

In [ ]:
! qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ordination_ileum-tgTRF.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-ileum-tgTRF.txt\
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-tgTRF_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-tgTRF_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-ileum-tgTRF.txt \
    --m-metadata-column ATTRIBUTE_ZT_collection_time \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-tgTRF-ZT-significance.qzv

### Deicode (to calculate aitchison) Ileum Only Tg-ALF

In [ ]:
! biom convert \
  -i /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-tgALF.txt \
  -o /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-tgALF.biom \
  --table-type="OTU table" \
  --to-hdf5

! qiime tools import \
  --input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-tgALF.biom \
  --output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-tgALF.qza \
  --type FeatureTable[Frequency]

In [ ]:
! qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-tgALF.qza \
    --p-min-feature-count 10 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ordination_ileum-tgALF.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-tgALF_lcms.qza

In [ ]:
! qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ordination_ileum-tgALF.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-ileum-tgALF.txt\
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-tgALF_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-tgALF_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-ileum-tgALF.txt \
    --m-metadata-column ATTRIBUTE_ZT_collection_time \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-tgALF-ZT-significance.qzv

### Deicode (to calculate aitchison) Ileum Only NTg-ALF

In [ ]:
! biom convert \
  -i /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-ntgALF.txt \
  -o /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-ntgALF.biom \
  --table-type="OTU table" \
  --to-hdf5

! qiime tools import \
  --input-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-ntgALF.biom \
  --output-path /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-ntgALF.qza \
  --type FeatureTable[Frequency]

In [ ]:
! qiime deicode rpca \
    --i-table /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/quantification_table/quantification_table-ileum-ntgALF.qza \
    --p-min-feature-count 10 \
    --p-min-sample-count 500 \
    --o-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ordination_ileum-ntgALF.qza \
    --o-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-ntgALF_lcms.qza

In [ ]:
! qiime emperor biplot \
    --i-biplot /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ordination_ileum-ntgALF.qza \
    --m-sample-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-ileum-ntgALF.txt\
    --p-ignore-missing-samples \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-ntgALF_biplot.qzv

In [ ]:
qiime diversity beta-group-significance \
    --i-distance-matrix /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-ntgALF_lcms.qza \
    --m-metadata-file /mnt/zarrinpar/scratch/sfloresr/AD_EM/data_files/FBMN/metadata_table/metadata_table-ileum-ntgALF.txt \
    --m-metadata-column ATTRIBUTE_ZT_collection_time \
    --p-method permanova \
    --p-pairwise \
    --o-visualization /mnt/zarrinpar/scratch/sfloresr/AD_EM/diversity_metrics/ileum/beta_deicode_ileum-ntgALF-ZT-significance.qzv